# AIHub CounselingSpeech (KtelSpeech) — EDA

`/data/ASR/RAW/AIHub_CounselingSpeech/012.상담_음성_데이터`

## 구조 (확인됨)
- **세션 폴더** `…/<라벨링|원천>데이터_1129_add/KtelSpeech_<split>_D##_<label|wav>_0/D##/J##/S########/`
- 라벨링: 발화별 `0001.txt`(UTF-8 전사) + 세션 매니페스트 `S########.json`
- 원천: 같은 트리에 `0001.wav` (오디오)
- **오디오 경로 매핑**: 라벨링데이터→원천데이터, `_label_0`→`_wav_0`

## JSON 스키마 (= 고객응대 데이터와 동일)
- `dataSet.typeInfo.category` (도메인), `.speakers[].{id,gender(여/남),type(상담원/고객),age,residence}`, `.inputType`
- `dataSet.dialogs[].{speaker(id), audioPath, textPath}` — 발화 순서·화자 매핑

## 유의
- ⚠️ **상담 대화 → 전사에 PII(이름 등) 포함** 가능. 처리·공유 주의.
- ⚠️ **전화망 가능성 → sr 8kHz 여부 확인**(오디오 셀). 16k와 섞을 때 리샘플 전략 결정.
- 2자 대화(상담원+고객). 화자 메타 풍부 → 분포·화자 분석 가능.

In [1]:
from pathlib import Path
import re, json, random, numpy as np, pandas as pd
from collections import Counter
from IPython.display import Audio, display
try:
    import soundfile as sf
except ImportError:
    sf = None
    print("⚠ soundfile 없음 → pip install soundfile")

ROOT = Path("/data/ASR/RAW/AIHub_CounselingSpeech/012.상담_음성_데이터")

def detect_split(p):
    s = str(p).lower()
    return "train" if "train" in s else "valid" if "valid" in s else "test" if "test" in s else "unknown"

def label_to_audio(p):
    # 라벨 경로 → 오디오 경로 (병렬 트리)
    return Path(str(p).replace("라벨링데이터", "원천데이터").replace("_label_", "_wav_"))

def read_txt(p):
    return Path(p).read_text(encoding="utf-8", errors="replace").strip()

print("ROOT 존재:", ROOT.is_dir())

ROOT 존재: True


## 1. 매니페스트 파서

> 세션 JSON을 돌며 dialogs를 펼쳐 발화 단위 manifest 생성. 전사는 옆 `.txt`, 오디오는 병렬 트리에서 해석. 화자 메타(gender/type/age/residence)를 붙임.
> 처음엔 `MAX_SESSIONS`로 일부만 빠르게 확인하고, 전체는 None으로.

In [2]:
MAX_SESSIONS = 300   # 빠른 1차 확인용. 전체 돌릴 땐 None

def build_manifest(root, max_sessions=MAX_SESSIONS):
    rows = []
    jsons = sorted(root.rglob("S*.json"))     # 세션 매니페스트
    if max_sessions:
        jsons = jsons[:max_sessions]
    for jp in jsons:
        try:
            ds = json.loads(jp.read_text(encoding="utf-8", errors="replace"))["dataSet"]
        except Exception:
            continue
        ti = ds.get("typeInfo", {})
        spk = {s["id"]: s for s in ti.get("speakers", [])}
        domain, split, session = ti.get("category"), detect_split(jp), jp.stem
        audio_dir = label_to_audio(jp.parent)
        for d in ds.get("dialogs", []):
            sid = d.get("speaker"); sm = spk.get(sid, {})
            txt_name = Path(d["textPath"]).name
            wav_name = Path(d["audioPath"]).name
            tf = jp.parent / txt_name
            rows.append({
                "split": split, "domain": domain, "session": session,
                "utt": Path(txt_name).stem, "speaker": sid,
                "spk_type": sm.get("type"), "gender": sm.get("gender"),
                "age": sm.get("age"), "residence": sm.get("residence"),
                "text": read_txt(tf) if tf.exists() else "",
                "audio_path": str(audio_dir / wav_name),
            })
    df = pd.DataFrame(rows)
    if len(df):
        df["text"] = df["text"].astype("object")
    return df

df = build_manifest(ROOT)
print(f"세션 {df['session'].nunique():,} / 발화 {len(df):,}  (MAX_SESSIONS={MAX_SESSIONS})")
print("split:", df["split"].value_counts().to_dict())
df.head()

세션 269 / 발화 15,064  (MAX_SESSIONS=300)
split: {'train': 15064}


,split,domain,session,utt,speaker,spk_type,gender,age,residence,text,audio_path
0,train,교육,S00000001,0001,9855,상담원,남,20대,서울,안녕하세요. 쉐어링 스터디 상담원 @이주빈입니다.,/data/ASR/RAW/AIHub_CounselingSpeech/012.상담_음성...
1,train,교육,S00000001,0002,tczpppab,고객,남,60대,경기,예 안녕하세요 제가 다른 게 아니라 나이가 좀 있는 지라 강좌를 결재 방법이 조금 ...,/data/ASR/RAW/AIHub_CounselingSpeech/012.상담_음성...
2,train,교육,S00000001,0003,tczpppab,고객,남,60대,경기,그거 (SHARING)/(쉐어링) (CASH)/(캐시)충전이었나,/data/ASR/RAW/AIHub_CounselingSpeech/012.상담_음성...
3,train,교육,S00000001,0004,tczpppab,고객,남,60대,경기,그게 처음에 (MY PAGE)/(마이 페이지)를 들어가서 (CASH)/(캐시)를 버...,/data/ASR/RAW/AIHub_CounselingSpeech/012.상담_음성...
4,train,교육,S00000001,0005,9855,상담원,남,20대,서울,(CASH)/(캐시) 할인권 (MENU)/(메뉴)에서 (SHARING CASH)/(...,/data/ASR/RAW/AIHub_CounselingSpeech/012.상담_음성...


In [3]:
# 매칭/결측 점검 (표본)
chk = df.sample(min(300, len(df)), random_state=0)
miss_audio = int(chk["audio_path"].map(lambda p: not Path(p).exists()).sum())
print(f"표본 {len(chk)} 중 오디오 없음: {miss_audio}")
print(f"빈 전사: {int((df['text'].str.strip()=='').sum())}건")
print(f"화자 메타 결측: gender {df['gender'].isna().sum()} / type {df['spk_type'].isna().sum()}")

표본 300 중 오디오 없음: 0
빈 전사: 0건
화자 메타 결측: gender 0 / type 0


## 2. 분포

In [4]:
df["text_len"] = df["text"].str.len()
for col in ["spk_type", "gender", "age", "residence", "domain", "split"]:
    print(f"=== {col} ===")
    print(df[col].value_counts(dropna=False).to_string())
    print()
print("=== 전사 글자 수 ===")
print(df["text_len"].describe().round(1).to_string())

=== spk_type ===
spk_type
고객     11332
상담원     3732

=== gender ===
gender
여    9682
남    5382

=== age ===
age
60대    4810
50대    2920
20대    2403
30대    1511
10대    1354
70대    1194
40대     872

=== residence ===
residence
서울    7995
경기    5401
대구     858
인천     810

=== domain ===
domain
교육    15064

=== split ===
split
train    15064

=== 전사 글자 수 ===
count    15064.0
mean        37.5
std         26.8
min          1.0
25%         18.0
50%         31.0
75%         50.0
max        236.0


## 3. 전사 컨벤션

> 깨끗한 문장체로 보이나, KsponSpeech식 태그·이중전사·특수기호 유무를 전수로 확인.

In [5]:
txt = df["text"].fillna("")
print("특수문자 전수 (상위 30):")
print(txt.str.findall(r"[^가-힣a-zA-Z0-9\s]").explode().value_counts().head(30).to_string())
print(f"\n영문 포함: {txt.str.contains(r'[A-Za-z]').mean()*100:.2f}%  /  숫자 포함: {txt.str.contains(r'[0-9]').mean()*100:.2f}%")

print("\nKsponSpeech식 주석 점검:")
hit = False
for tag in ["b/", "n/", "l/", "o/", "u/", ")/("]:
    c = int(txt.str.contains(re.escape(tag)).sum())
    if c:
        print(f"  '{tag}': {c:,}건"); hit = True
if not hit:
    print("  없음")

print("\n전사 샘플 8개:")
for t in txt.sample(min(8, len(txt)), random_state=1):
    print("  •", t[:80])

특수문자 전수 (상위 30):
text
/    12519
(    11806
)    11804
.    10857
?     6063
,     1736
+      678
*      257
@      223
%       72
-       67
:        7
!        7
[        3
]        3
&        2
~        2
        1
ㅡ        1
'        1
ㄴ        1
…        1

영문 포함: 36.36%  /  숫자 포함: 12.29%

KsponSpeech식 주석 점검:
  'b/': 199건
  'n/': 4,036건
  'l/': 11건
  'o/': 219건
  'u/': 123건
  ')/(': 3,816건

전사 샘플 8개:
  • 마침 제가 셰어링 카드를 이용하고 있어서 그 카드로 결재를 하려고 했는데 (10)(십)%나 할인 받을 수 있다니
  • (25년)/(이십 오 년) (11월)/(십일 월)이에요.
  • n/ 별말씀을요.
  • n/ 네, 감사합니다. 그럼 문자 기다리면 되고 이제 끊어도 되는 거죠?
  • 제가 핸드폰으로 가끔 동영상 볼 때 튕기거나 버퍼링이 발생한 경우가 있던데 만약 강의를 수강 중에 이런 일이 발생할 경우 조치 사항이 있으면 알
  • 네 고객님 성함이 어떻게 되시죠?
  • 오늘도 아이디가 생각이 안나서 여러 번 시도하다 안되서 전화드린거거든요. 변경할 수 있을까요?
  • 그러면 뭐 신청하려고 하는 강좌명을 선택한 후 오른쪽 장바구니 (image)/(이미지) 같은게 있던데 그거 눌러서 담게 하면 돼요?


## 3-1. 실제 음성 청취

In [6]:
# 화자 타입과 함께 청취 (상담원/고객 번갈아)
print("샘플 청취:")
for r in df.sample(3, random_state=1).itertuples():
    print(f"[{r.spk_type}/{r.gender}] {r.text[:70]}")
    if sf and Path(r.audio_path).exists():
        data, sr = sf.read(r.audio_path)
        display(Audio(data, rate=sr))
    else:
        print("   (오디오 없음 또는 soundfile 미설치)")

샘플 청취:
[고객/여] 마침 제가 셰어링 카드를 이용하고 있어서 그 카드로 결재를 하려고 했는데 (10)(십)%나 할인 받을 수 있다니


[고객/여] (25년)/(이십 오 년) (11월)/(십일 월)이에요.


[상담원/여] n/ 별말씀을요.


## 4. 오디오 속성 — ⚠️ 8kHz 여부 확인 (전화망 핵심)

In [7]:
paths = df["audio_path"].tolist()
samp = random.Random(0).sample(paths, min(100, len(paths)))
rows = []
for p in samp:
    if sf and Path(p).exists():
        try:
            i = sf.info(p); rows.append((i.samplerate, i.channels, i.subtype, round(i.frames/i.samplerate, 2)))
        except Exception as e:
            rows.append(("ERR", str(e)[:20], "", None))
a = pd.DataFrame(rows, columns=["sr", "ch", "subtype", "dur"])
print("표본", len(a), "개")
print("sample_rate :", a["sr"].value_counts().to_dict())
print("channels    :", a["ch"].value_counts().to_dict())
print("subtype     :", a["subtype"].value_counts().to_dict())
if len(a["dur"].dropna()):
    print(f"길이(초): 평균 {a['dur'].dropna().mean():.2f} / 최대 {a['dur'].dropna().max():.2f}")
print("\n⚠ sr=8000이면 전화망 → 16k 셋과 섞을 때 리샘플 결정. ch=2면 화자분리/믹스다운 결정.")

표본 100 개
sample_rate : {8000: 100}
channels    : {1: 100}
subtype     : {'PCM_16': 100}
길이(초): 평균 4.47 / 최대 13.02

⚠ sr=8000이면 전화망 → 16k 셋과 섞을 때 리샘플 결정. ch=2면 화자분리/믹스다운 결정.


## 5. 화자 분석

> 상담원 id는 숫자(재사용 가능), 고객 id는 해시형(콜마다 고유)일 수 있음 → 화자 타입별로 분리해 본다.

In [8]:
by_type = df.groupby("spk_type")["speaker"].nunique()
print("화자 타입별 고유 화자 수:")
print(by_type.to_string())

# split이 둘 이상이면(train/valid 함께 로드 시) 타입별 화자 누수 점검
if df["split"].nunique() > 1:
    for t in df["spk_type"].dropna().unique():
        sub = df[df.spk_type == t]
        tr = set(sub[sub.split=="train"]["speaker"]); va = set(sub[sub.split=="valid"]["speaker"])
        if tr and va:
            print(f"  [{t}] train∩valid 화자 겹침: {len(tr&va)} / valid {len(va)}")
print("\n화자별 발화 수: 평균 %.0f / 최대 %d" % (df["speaker"].value_counts().mean(), df["speaker"].value_counts().max()))

화자 타입별 고유 화자 수:
spk_type
고객     39
상담원    23

화자별 발화 수: 평균 243 / 최대 783


In [9]:
# ============================================================
# 비식별화(PII) 토큰 포함 발화 → 전사 + 음성 직접 청취  (모든 EDA 노트북 공용)
# 파서로 DataFrame을 만든 셀을 먼저 실행한 뒤, 이 셀을 새 셀에 붙여 실행.
# DataFrame 변수(df/df_v/...)와 오디오 경로 컬럼(audio_path/wav/...)을 자동 탐지.
# ============================================================
import re
from pathlib import Path
from IPython.display import Audio, display
try:
    import soundfile as sf
except ImportError:
    sf = None
    print("⚠ soundfile 없음 → conda activate TRAIN-ASR")

import pandas as pd

# ---------- 설정 ----------
N_LISTEN = 5          # 들어볼 발화 수
RANDOM_STATE = 0      # None이면 매번 다른 표본
# 비식별화(PII) 토큰: 음향 토큰(b/ n/ l/ o/ u/)과 구분되는 익명화 전용 패턴
PII_PATTERNS = {
    "@ (이름 마커)":           re.compile(r"@"),
    "ㅇㅇ류 (익명화 2자+)":     re.compile(r"ㅇ{2,}"),
    "name/ (이름 태그)":        re.compile(r"(?:^|\s)name/", re.I),
    "[마스킹]":                re.compile(r"\[[^\]]{0,15}\]"),
    "<마스킹>":                re.compile(r"<[^>]{0,15}>"),
    "*** (별표 2+)":           re.compile(r"\*{2,}"),
    "xxx (엑스 2+)":           re.compile(r"[xX]{2,}"),
    "○○ (공백원 2+)":          re.compile(r"[○◯]{2,}"),
}

# ---------- 1) text DataFrame 자동 탐지 ----------
def _find_text_frame():
    g = globals()
    for name in ["df_v","df_valid","df","df_t","df_train","d","a"]:
        o = g.get(name)
        if isinstance(o, pd.DataFrame) and "text" in o.columns:
            return name, o
    for name, o in g.items():
        if not name.startswith("_") and isinstance(o, pd.DataFrame) and "text" in o.columns:
            return name, o
    return None, None

_name, _df = _find_text_frame()
if _df is None:
    raise RuntimeError("text 컬럼 DataFrame 없음 — 파서 셀을 먼저 실행하세요.")

# ---------- 2) 오디오 경로 컬럼 자동 탐지 ----------
AUDIO_COL = next((c for c in ["audio_path","wav","src_wav","audio","path","filepath"]
                  if c in _df.columns), None)
print(f"[대상] DataFrame '{_name}' · {len(_df):,} 발화 · 오디오 컬럼: {AUDIO_COL or '없음(PCM 직접 노트북일 수 있음)'}\n")

# ---------- 3) PII 토큰 집계 ----------
_txt = _df["text"].fillna("").astype("object")
print("=== 비식별화 토큰 집계 (text 원본) ===")
present = []
for label, pat in PII_PATTERNS.items():
    occ = int(_txt.str.count(pat).sum())
    utt = int(_txt.str.contains(pat).sum())
    if occ:
        present.append((label, pat, utt, occ))
        print(f"  {label:20s} 발화 {utt:>6,} · 출현 {occ:>6,} ({utt/len(_df)*100:.3f}%)")
if not present:
    print("  → 이 데이터셋엔 정의된 비식별화 토큰이 없음 (자유발화/낭독 등). 청취 생략.")

# ---------- 4) PII 포함 발화만 필터 → 전사 + 음성 재생 ----------
if present and AUDIO_COL:
    mask = pd.Series(False, index=_df.index)
    for _, pat, _, _ in present:
        mask |= _txt.str.contains(pat)
    hits = _df[mask]
    print(f"\n=== 비식별화 토큰 포함 발화 {len(hits):,}건 중 {min(N_LISTEN,len(hits))}개 청취 ===")
    print("   (전사의 마스킹 부분이 음성에서 실제로 어떻게 발화되는지 직접 확인)\n")
    sample = hits.sample(min(N_LISTEN, len(hits)), random_state=RANDOM_STATE)
    for r in sample.itertuples():
        text = getattr(r, "text", "")
        ap = getattr(r, AUDIO_COL, None)
        # 어떤 PII 패턴에 걸렸는지 표시
        tags = [lab for lab, pat, _, _ in present if pat.search(text or "")]
        print(f"[{', '.join(tags)}]")
        print(f"  전사: {text[:100]}")
        if sf and ap and Path(str(ap)).exists():
            try:
                data, sr = sf.read(str(ap))
                display(Audio(data, rate=sr))
            except Exception as e:
                print(f"   (재생 실패: {str(e)[:50]})")
        else:
            print(f"   (오디오 경로 없음/미존재: {ap})")
        print()
elif present and not AUDIO_COL:
    print("\n⚠ PII 토큰은 있으나 오디오 경로 컬럼을 못 찾음.")
    print("  이 노트북이 PCM을 직접 읽는 방식이면, 아래처럼 수동 지정:")
    print("  → sample = _df[mask].sample(N_LISTEN); 각 행의 키로 원본 PCM 경로를 구성해 재생")

[대상] DataFrame 'df' · 15,064 발화 · 오디오 컬럼: audio_path

=== 비식별화 토큰 집계 (text 원본) ===
  @ (이름 마커)            발화    127 · 출현    223 (0.843%)
  [마스킹]                발화      2 · 출현      3 (0.013%)

=== 비식별화 토큰 포함 발화 129건 중 5개 청취 ===
   (전사의 마스킹 부분이 음성에서 실제로 어떻게 발화되는지 직접 확인)

[@ (이름 마커)]
  전사: 그러면 직장으로 배송 부탁드릴께요. @서울시 @영등포구 @직장로 @(112)/(일 일 이) @코로나 싫어 빌딩 @(12층)/(십 이 층) @(505호)/(오백 오 호)에요.



[@ (이름 마커)]
  전사: n/ @서울 @송파구 @송파동 @(123-23번지)/(백 이십 삼 다시 이십 삼 번지) @좋은아파트 @(101동)/(백 일 동) @(1호)/(일 호)입니다. 최근에 주소 변경도 없



[@ (이름 마커)]
  전사: o/ 감사합니다. 상담원 @김나봉이었습니다.



[@ (이름 마커)]
  전사: @호유입니다.



[@ (이름 마커)]
  전사: @최라라이고요. 생일은 @(95년)/(구십 오 년) @(1월)/(일 월) @(12일)/(십 이 일)이에요.


In [10]:
import json
from pathlib import Path

src = '/data/ASR/BENCHMARK/SILVER/AIHub_CounselingSpeech/valid/transcript.jsonl'
shown = 0
with open(src, encoding='utf-8') as f:
    for line in f:
        if not line.strip():
            continue
        d = json.loads(line)
        if not str(d.get('text_norm') or '').strip():
            print(f"audio={d['audio']}  text={d.get('text')!r}  text_norm={d.get('text_norm')!r}")
            shown += 1
            if shown >= 10:
                break

audio=audio/S00000539_0022.wav  text='u/'  text_norm=''
audio=audio/S00013265_0022.wav  text='u/'  text_norm=''


In [11]:
import json
from collections import Counter

src = '/data/ASR/BENCHMARK/SILVER/AIHub_CounselingSpeech/valid/transcript.jsonl'
c = Counter()
with open(src, encoding='utf-8') as f:
    for line in f:
        if not line.strip():
            continue
        d = json.loads(line)
        if not str(d.get('text_norm') or '').strip():
            c[str(d.get('text'))] += 1
print(c.most_common())

[('u/', 2)]
